# Search Laboratory: A* on a Warehouse Grid (Using an LLM as an Engineering Assistant)
This notebook is a worked solution and report for the lab *Laboratory Exercise: Search and A\**.

**How to use it:** run *Kernel → Restart & Run All*. It needs only the Python standard library (`heapq`, `collections`, `math`, `random`), so it takes a few seconds. All numbers quoted in the write-up come from the code cells; the experiments are deterministic (fixed seeds, fixed tie-breaking).

| Section | Content |
|---|---|
| Task 0 | Formulate the warehouse problem as a search problem |
| Task 1 | Design of the agent (before any code) |
| Task 2 | LLM prompt and the A* implementation |
| Task 3 | Systematic tests (plus seeded-bug tests) |
| Task 4 | Where each concept appears in the code |
| Task 5 | A* vs. BFS |
| Task 6 | Heuristic investigation |
| Task 7 | Evaluation of the LLM-generated agent |
| - | Final reflection, prompt appendix |

**Conventions used throughout.** A state is a grid position `(row, col)`, row 0 at the top. The goal test is applied when a state is *removed* from the frontier, for both BFS and A*. "States expanded" means the number of times a non-goal state is removed from the frontier and its successors are generated, so both algorithms are counted the same way.

## Task 0: Understand the search problem

Before any code: the warehouse as $P=(S,A,T,s_0,G,c)$.

| Component | Specification |
|---|---|
| **State $S$** | The robot's grid cell $(r,c)$ with $0\le r<9,\ 0\le c<17$ that is not an obstacle. (The map is static and known, so it is part of the problem, not of the state.) |
| **Actions $A$** | $\{\text{Up},\text{Down},\text{Left},\text{Right}\}$, i.e. $(\Delta r,\Delta c)\in\{(-1,0),(1,0),(0,-1),(0,1)\}$ |
| **Transition $T$** | $T((r,c),a)=(r+\Delta r_a,\ c+\Delta c_a)$ if that cell is inside the map and is not `#`; otherwise the action is not applicable |
| **Initial state $s_0$** | The cell marked `S`, which is $(1,1)$ |
| **Goal $G$** | $\{$the cell marked `G`$\}=\{(7,15)\}$ |
| **Cost $c$** | $c(s,a,s')=1$ for every move, so path cost = number of moves |

**(a) What information is necessary to specify a state?** Only the robot's position $(r,c)$. The obstacle layout never changes and the goal is fixed, so nothing else is needed to predict what the robot can do next. (If obstacles moved, or the robot had to pick up items or had a battery level, those would have to be added to the state.)

**(b) What makes an action invalid?** Moving outside the map, or moving into a `#` cell. (The outer wall makes the first case impossible in this map, but the code still checks bounds.)

**(c) Is this a deterministic search problem?** Yes. The environment is fully observable, static, discrete and known, and each action in a state has exactly one outcome, so the agent can compute the whole plan in advance before moving.

**(d) What would constitute a solution?** A sequence of valid actions $a_1,\dots,a_k$ that takes the robot from $s_0=(1,1)$ to $(7,15)$; equivalently a path of adjacent free cells. An *optimal* solution is one with the smallest $k$ (since all costs are 1).

**Think about it.** If the state, transition and goal cannot be stated precisely, then code generated by an LLM has nothing to be checked against; formulating the problem is what makes testing possible later.

## Task 1: Plan the agent (written before prompting the LLM)

1. **State representation in Python:** a tuple `(row, col)`. Tuples are immutable and hashable, so they can be used as dictionary keys (for the "reached" table and parent pointers).
2. **Warehouse representation:** the ASCII map is parsed into a list of lists of characters (`grid[row][col]`); `S` and `G` are located during parsing and remembered as `start` and `goal`.
3. **Valid actions:** a function `successors(grid, state)` tries the four moves `Up/Down/Left/Right`, and keeps a move only if the new cell is inside the grid and `grid[r][c] != '#'`. Each successor is reported as `(action_name, next_state, step_cost)`.
4. **Goal recognition:** `state == goal`, tested when a state is removed from the frontier (this is what guarantees optimality for A*).
5. **Frontier contents (A\*):** a priority queue (binary heap) of entries `(f, tie_breaker, g, state)`, where $f=g+h$ is the priority. The tie-breaker is an increasing counter so that equal-$f$ entries are served first-in-first-out and Python never has to compare states. `g` is stored so stale duplicates can be recognised.
6. **Path reconstruction:** a `parent` dictionary `child → parent state` filled in whenever a cheaper route to a state is found; at the goal we follow parents back to the start and reverse.

**What the program reports:** whether a solution was found, the path, the path length, and the number of states expanded (plus the path cost, which equals the length here).

**Extra design decisions.** (i) Keep `best_g[state]`, the cheapest known cost of reaching each state. A state is re-inserted only if a *cheaper* route is found, which keeps A* correct even for heuristics that are not consistent (Task 6). (ii) Keep the map, the heuristic and the algorithm separate, so the heuristic is a parameter and BFS can reuse the same `successors` function.

**Think about it.** This design is mine; the LLM is later asked only to translate it into code, and the code is then checked against this design in Task 4.

## Task 2: Ask an LLM to generate A*

**Prompt used** (built from the Task 1 design, based on the example prompt in the handout):

> I am implementing a simple goal-based search agent in Python. The environment is a grid represented by an ASCII map. The agent starts at `S` and must reach `G`. The symbols `#` represent obstacles and `.` represents free cells. The agent can move up, down, left, or right, and every movement has cost 1.
> Implement A* search. Use Manhattan distance as the heuristic, h(n) = |x − xG| + |y − yG|.
> Design requirements: represent states as `(row, col)` tuples; parse the map into a list of lists; write a separate `successors(grid, state)` function that yields `(action, next_state, cost)`; use `heapq` as the frontier with entries `(f, tie_breaker, g, state)`; keep a dictionary of the best known g for every state and a `parent` dictionary; test for the goal when a state is popped; do not expand the same state repeatedly (but allow a cheaper path to re-open a state); reconstruct the path; return whether a solution was found, the path, the path length, and the number of states expanded. Do not use any external libraries. Keep the implementation simple and explain the main components of the code.

*(If you used different wording, replace this block with your exact prompt; see also the appendix.)*

The code below is the implementation after inspection. It is **not assumed correct**: Tasks 3-4 test and inspect it.

In [1]:
import heapq, math, random
from collections import deque

WAREHOUSE = """
#################
#S....#.........#
#.###.#.#######.#
#...#.#.......#.#
###.#.#######.#.#
#...#.........#.#
#.###########.#.#
#.............#G#
#################
"""

def parse_map(text):
    """ASCII map -> (grid as list of lists, start state, goal state)."""
    grid = [list(row) for row in text.strip().splitlines()]
    start = goal = None
    for r, row in enumerate(grid):
        for c, ch in enumerate(row):
            if ch == "S": start = (r, c)
            elif ch == "G": goal = (r, c)
    return grid, start, goal

def render(grid, path=None):
    """Draw the map, marking the cells of `path` with '*'."""
    g2 = [row[:] for row in grid]
    for (r, c) in (path or []):
        if g2[r][c] == ".": g2[r][c] = "*"
    return "\n".join("".join(row) for row in g2)

# ---- problem definition: actions and transition model ---------------------------------
ACTIONS = {"Up": (-1, 0), "Down": (1, 0), "Left": (0, -1), "Right": (0, 1)}

def successors(grid, state):
    """Yield (action, next_state, step_cost) for every VALID action in `state`."""
    r, c = state
    for name, (dr, dc) in ACTIONS.items():
        nr, nc = r + dr, c + dc
        inside = 0 <= nr < len(grid) and 0 <= nc < len(grid[0])
        if inside and grid[nr][nc] != "#":
            yield name, (nr, nc), 1

# ---- heuristics --------------------------------------------------------------------------
def manhattan(s, goal): return abs(s[0] - goal[0]) + abs(s[1] - goal[1])
def euclidean(s, goal): return math.hypot(s[0] - goal[0], s[1] - goal[1])
def zero(s, goal):      return 0

def reconstruct(parent, state):
    path = []
    while state is not None:
        path.append(state)
        state = parent[state]
    return path[::-1]

# ---- A* ---------------------------------------------------------------------------------------
def astar(grid, start, goal, h=manhattan):
    tie = 0
    frontier = [(h(start, goal), tie, 0, start)]     # (f, tie-breaker, g, state)
    best_g = {start: 0}                              # cheapest known cost to reach each state
    parent = {start: None}
    expanded, expanded_states = 0, set()
    while frontier:
        f, _, g_s, s = heapq.heappop(frontier)
        if g_s > best_g[s]:                          # stale entry: a cheaper route was found later
            continue
        if s == goal:                                # goal test on removal from the frontier
            path = reconstruct(parent, s)
            return {"found": True, "path": path, "length": len(path) - 1, "cost": g_s,
                    "expanded": expanded, "distinct": len(expanded_states)}
        expanded += 1
        expanded_states.add(s)
        for action, s2, step_cost in successors(grid, s):
            g2 = g_s + step_cost                     # g(n')
            if s2 not in best_g or g2 < best_g[s2]:
                best_g[s2] = g2
                parent[s2] = s
                tie += 1
                heapq.heappush(frontier, (g2 + h(s2, goal), tie, g2, s2))   # f(n') = g(n') + h(n')
    return {"found": False, "path": None, "length": None, "cost": None,
            "expanded": expanded, "distinct": len(expanded_states)}

def actions_of(path):
    """Translate a list of states into the list of actions the robot executes."""
    names = {v: k for k, v in ACTIONS.items()}
    return [names[(b[0] - a[0], b[1] - a[1])] for a, b in zip(path, path[1:])]

def check_path(grid, start, goal, path):
    """Independent validity check: starts at S, ends at G, unit steps, never enters '#'."""
    if not path or path[0] != start or path[-1] != goal: return False
    for a, b in zip(path, path[1:]):
        if abs(a[0] - b[0]) + abs(a[1] - b[1]) != 1 or grid[b[0]][b[1]] == "#": return False
    return True

print("Definitions loaded.")

Definitions loaded.


## Task 3: Test the generated program

### Test 1: the original warehouse

In [2]:
grid, start, goal = parse_map(WAREHOUSE)
res = astar(grid, start, goal)             # Manhattan heuristic (default)

print("start:", start, " goal:", goal)
print("path found:      ", res["found"])
print("path length:     ", res["length"], "moves (cost", res["cost"], ")")
print("states expanded: ", res["expanded"])
print("path valid (independent check):", check_path(grid, start, goal, res["path"]))
print("\npath as states:\n", res["path"])
print("\npath as actions:\n", actions_of(res["path"]))
print("\nmap with path (*):")
print(render(grid, res["path"]))

warehouse_len, warehouse_exp = res["length"], res["expanded"]

start: (1, 1)  goal: (7, 15)
path found:       True
path length:      40 moves (cost 40 )
states expanded:  63
path valid (independent check): True

path as states:
 [(1, 1), (1, 2), (1, 3), (1, 4), (1, 5), (2, 5), (3, 5), (4, 5), (5, 5), (5, 6), (5, 7), (5, 8), (5, 9), (5, 10), (5, 11), (5, 12), (5, 13), (4, 13), (3, 13), (3, 12), (3, 11), (3, 10), (3, 9), (3, 8), (3, 7), (2, 7), (1, 7), (1, 8), (1, 9), (1, 10), (1, 11), (1, 12), (1, 13), (1, 14), (1, 15), (2, 15), (3, 15), (4, 15), (5, 15), (6, 15), (7, 15)]

path as actions:
 ['Right', 'Right', 'Right', 'Right', 'Down', 'Down', 'Down', 'Down', 'Right', 'Right', 'Right', 'Right', 'Right', 'Right', 'Right', 'Right', 'Up', 'Up', 'Left', 'Left', 'Left', 'Left', 'Left', 'Left', 'Up', 'Up', 'Right', 'Right', 'Right', 'Right', 'Right', 'Right', 'Right', 'Right', 'Down', 'Down', 'Down', 'Down', 'Down', 'Down']

map with path (*):
#################
#S****#*********#
#.###*#*#######*#
#...#*#*******#*#
###.#*#######*#*#
#...#*********#*#
#.##

### Tests 2-4: trivial case, no solution, alternative paths
Expected behaviour is known in advance for each map (written down *before* running):

| Test | Map | Expected |
|---|---|---|
| 2 trivial | goal adjacent to start | found, length 1 |
| 3 no solution | goal walled off | `found = False`, terminates |
| 4a alternative paths | two routes (lengths 5 and 7) | shortest route, length 5 |
| 4b many equal routes | open 3×3 room, S and G at opposite corners | length 4 (Manhattan distance) |

In [3]:
TRIVIAL = """
#####
#SG##
#####
"""
NO_SOLUTION = """
#######
#S....#
###.###
#...#G#
#######
"""
ALTERNATIVES = """
#######
#S....#
#.###.#
#...G.#
#######
"""
TIE_ROOM = """
#####
#S..#
#...#
#..G#
#####
"""

def run_map(name, text, expected_found, expected_length=None):
    g, s, t = parse_map(text)
    r = astar(g, s, t)
    ok = (r["found"] == expected_found) and (expected_length is None or r["length"] == expected_length)
    if r["found"]:
        ok = ok and check_path(g, s, t, r["path"])
    print(f"{name:22s} found={r['found']!s:5s} length={r['length']!s:5s} expanded={r['expanded']:3d}  ->  {'PASS' if ok else 'FAIL'}")
    if r["found"]:
        print(render(g, r["path"]))
    print()
    assert ok, name
    return r

run_map("Test 2 trivial", TRIVIAL, True, 1)
run_map("Test 3 no solution", NO_SOLUTION, False)
run_map("Test 4a alternatives", ALTERNATIVES, True, 5)
run_map("Test 4b tie room", TIE_ROOM, True, 4)

Test 2 trivial         found=True  length=1     expanded=  1  ->  PASS
#####
#SG##
#####

Test 3 no solution     found=False length=None  expanded=  9  ->  PASS

Test 4a alternatives   found=True  length=5     expanded=  8  ->  PASS
#######
#S....#
#*###.#
#***G.#
#######

Test 4b tie room       found=True  length=4     expanded=  8  ->  PASS
#####
#S..#
#*..#
#**G#
#####



Test 3 confirms that the algorithm terminates with failure: the walled-off goal means the frontier eventually runs empty (the `best_g` table stops states from being re-added forever). Test 4a shows the robot takes the 5-move route (down the left side) rather than the 7-move route along the top.

### Stronger evidence: cross-validation on random maps
Four hand-made maps prove little. BFS is a trusted *reference* for shortest paths in a unit-cost grid (and its code is much simpler), so we compare A* with BFS on 300 random 8×8 maps, and also validate every returned path independently.

In [4]:
def bfs(grid, start, goal):
    frontier = deque([start])
    parent = {start: None}
    expanded = 0
    while frontier:
        s = frontier.popleft()
        if s == goal:                               # same convention as A*: test on removal
            path = reconstruct(parent, s)
            return {"found": True, "path": path, "length": len(path) - 1, "cost": len(path) - 1,
                    "expanded": expanded, "distinct": expanded}
        expanded += 1
        for action, s2, step_cost in successors(grid, s):
            if s2 not in parent:                    # 'reached' table: never add a state twice
                parent[s2] = s
                frontier.append(s2)
    return {"found": False, "path": None, "length": None, "cost": None, "expanded": expanded, "distinct": expanded}

def random_map(rng, rows, cols, density):
    grid = [["#"] * cols for _ in range(rows)]
    free = []
    for r in range(1, rows - 1):
        for c in range(1, cols - 1):
            if rng.random() >= density:
                grid[r][c] = "."; free.append((r, c))
    if len(free) < 2: return None
    s, t = rng.sample(free, 2)
    grid[s[0]][s[1]] = "S"; grid[t[0]][t[1]] = "G"
    return grid, s, t

def make_maps(seed, n, rows, cols, density):
    rng, maps = random.Random(seed), []
    while len(maps) < n:
        m = random_map(rng, rows, cols, density)
        if m: maps.append(m)
    return maps

RANDOM_MAPS = make_maps(seed=1, n=300, rows=8, cols=8, density=0.25)

solvable = mismatches = invalid = 0
for g, s, t in RANDOM_MAPS:
    ra, rb = astar(g, s, t), bfs(g, s, t)
    solvable += rb["found"]
    if ra["found"] != rb["found"] or ra["length"] != rb["length"]: mismatches += 1
    if ra["found"] and not check_path(g, s, t, ra["path"]): invalid += 1

print(f"random maps: {len(RANDOM_MAPS)}  (solvable: {solvable}, unsolvable: {len(RANDOM_MAPS) - solvable})")
print("A* vs BFS disagreements (found flag or path length):", mismatches)
print("invalid A* paths:", invalid)
assert mismatches == 0 and invalid == 0

random maps: 300  (solvable: 261, unsolvable: 39)
A* vs BFS disagreements (found flag or path length): 0
invalid A* paths: 0


### Seeded-bug tests: does the test suite actually catch mistakes?
"Working output ≠ validated algorithm." A plausible path can come from a wrong algorithm. To see which tests matter, we deliberately write three *plausible* faulty versions of the kind an LLM (or a student) might produce and run them through the same tests:

1. **greedy**: priority uses `h` only (forgets `g`), so it is greedy best-first search, not A*;
2. **closed-on-push**: marks a state as "visited" when it is *generated*, so a cheaper route found later is ignored;
3. **no-closed-set**: a tree search that never remembers visited states, so it can loop forever on graphs with cycles (a cap of expansions turns an endless loop into a detectable failure).

Two more maps are added to the suite; they were found by random search and are "trap" maps where these bugs give a longer path.

In [5]:
TRAP = """
#######
#G#.#.#
#.#...#
#.#.#S#
#.....#
#..#..#
#######
"""

def greedy_bug(grid, start, goal):
    tie, frontier, parent, closed = 0, [(manhattan(start, goal), 0, start)], {start: None}, set()
    while frontier:
        _, _, s = heapq.heappop(frontier)
        if s in closed: continue
        closed.add(s)
        if s == goal:
            p = reconstruct(parent, s); return {"found": True, "path": p, "length": len(p) - 1}
        for a, s2, c in successors(grid, s):
            if s2 not in closed and s2 not in parent:
                parent[s2] = s; tie += 1
                heapq.heappush(frontier, (manhattan(s2, goal), tie, s2))          # BUG: no g(n)
    return {"found": False, "path": None, "length": None}

def closed_on_push_bug(grid, start, goal):
    tie, frontier, parent, seen = 0, [(manhattan(start, goal), 0, 0, start)], {start: None}, {start}
    while frontier:
        _, _, g_s, s = heapq.heappop(frontier)
        if s == goal:
            p = reconstruct(parent, s); return {"found": True, "path": p, "length": len(p) - 1}
        for a, s2, c in successors(grid, s):
            if s2 not in seen:                                                     # BUG: first route wins
                seen.add(s2); parent[s2] = s; tie += 1
                heapq.heappush(frontier, (g_s + c + manhattan(s2, goal), tie, g_s + c, s2))
    return {"found": False, "path": None, "length": None}

def no_closed_bug(grid, start, goal, cap=3000):
    tie, frontier, expanded = 0, [(manhattan(start, goal), 0, 0, (start, None))], 0
    while frontier:
        _, _, g_s, node = heapq.heappop(frontier)
        s = node[0]
        if s == goal:
            p = []
            while node: p.append(node[0]); node = node[1]
            return {"found": True, "path": p[::-1], "length": len(p) - 1}
        expanded += 1
        if expanded > cap: raise RuntimeError("did not terminate")
        for a, s2, c in successors(grid, s):                                       # BUG: no visited check
            tie += 1
            heapq.heappush(frontier, (g_s + c + manhattan(s2, goal), tie, g_s + c, (s2, node)))
    return {"found": False, "path": None, "length": None}

SUITE = [("T1 warehouse", WAREHOUSE, True, 40), ("T2 trivial", TRIVIAL, True, 1),
         ("T3 no solution", NO_SOLUTION, False, None), ("T4a alternatives", ALTERNATIVES, True, 5),
         ("T4b tie room", TIE_ROOM, True, 4), ("T4c trap map", TRAP, True, 8)]

def grade(solver):
    out = {}
    for name, text, exp_found, exp_len in SUITE:
        g, s, t = parse_map(text)
        try:
            r = solver(g, s, t)
            ok = r["found"] == exp_found and (exp_len is None or r["length"] == exp_len)
            ok = ok and (not r["found"] or check_path(g, s, t, r["path"]))
        except RuntimeError:
            ok = False
        out[name] = ok
    bad = 0
    for g, s, t in RANDOM_MAPS:
        ref = bfs(g, s, t)
        try:
            r = solver(g, s, t)
            if r["found"] != ref["found"] or r["length"] != ref["length"]: bad += 1
        except RuntimeError:
            bad += 1
    out["random x300"] = (bad == 0)
    out["_bad_random"] = bad
    return out

solvers = {"A* (correct)": astar, "greedy bug": greedy_bug,
           "closed-on-push bug": closed_on_push_bug, "no-closed-set bug": no_closed_bug}
grades = {name: grade(fn) for name, fn in solvers.items()}
cols = [k for k in grades["A* (correct)"] if not k.startswith("_")]
print(f"{'solver':20s}" + "".join(f"{c:>17s}" for c in cols))
for name, gr in grades.items():
    print(f"{name:20s}" + "".join(f"{('pass' if gr[c] else 'FAIL'):>17s}" for c in cols))
print("\nrandom-map disagreements with BFS:", {n: g['_bad_random'] for n, g in grades.items()})

assert all(v for k, v in grades["A* (correct)"].items() if not k.startswith("_"))
assert not all(v for k, v in grades["greedy bug"].items() if not k.startswith("_"))
assert not all(v for k, v in grades["closed-on-push bug"].items() if not k.startswith("_"))
assert not grades["no-closed-set bug"]["T3 no solution"]
assert grades["closed-on-push bug"]["T1 warehouse"] and not grades["greedy bug"]["T1 warehouse"]
g_, s_, t_ = parse_map(WAREHOUSE)
greedy_len = greedy_bug(g_, s_, t_)["length"]
trap_g, trap_s, trap_t = parse_map(TRAP)
cop_trap_len, opt_trap_len = closed_on_push_bug(trap_g, trap_s, trap_t)["length"], bfs(trap_g, trap_s, trap_t)["length"]
print(f"\ngreedy bug on the warehouse: {greedy_len} moves (optimal is 40)")
print(f"closed-on-push bug on the trap map: {cop_trap_len} moves (optimal is {opt_trap_len})")

solver                   T1 warehouse       T2 trivial   T3 no solution T4a alternatives     T4b tie room     T4c trap map      random x300
A* (correct)                     pass             pass             pass             pass             pass             pass             pass
greedy bug                       FAIL             pass             pass             pass             pass             FAIL             FAIL
closed-on-push bug               pass             pass             pass             pass             pass             FAIL             FAIL
no-closed-set bug                FAIL             pass             FAIL             pass             pass             pass             FAIL

random-map disagreements with BFS: {'A* (correct)': 0, 'greedy bug': 7, 'closed-on-push bug': 1, 'no-closed-set bug': 36}

greedy bug on the warehouse: 48 moves (optimal is 40)
closed-on-push bug on the trap map: 10 moves (optimal is 8)


**What the tests taught.**

- **Closed-on-push** passes the original warehouse and every hand-made map except the trap map (10 moves instead of 8), and it disagrees with BFS on only 1 of 300 random maps. It is a subtle bug: a run that only checks "a path was found" on the warehouse would accept it.
- **Greedy** looks plausible on small maps, but on the warehouse it returns a 48-move path where the optimum is 40. That is caught only because we knew the expected optimal length in advance and did not merely check that a path exists; it also fails the trap map and the random maps.
- **No-closed-set** search never terminates on the unsolvable map, and on the warehouse it revisits states so often that it exceeds the expansion cap.

So the trivial and "unsolvable" tests check *termination and boundary behaviour*, the trap, alternative-path and random tests check *optimality*, and the test expectations (known optimal lengths) are what make them meaningful. Only the combination gives confidence.

## Task 4: Inspect the A* algorithm

| Concept | Where it appears in the code |
|---|---|
| **State** | A `(row, col)` tuple, e.g. `start`, `s`, `s2`, `goal` |
| **Action** | The keys of `ACTIONS` (`"Up"`, `"Down"`, `"Left"`, `"Right"`), produced as the first element of each `successors()` result |
| **Transition** | `successors()`: `(nr, nc) = (r + dr, c + dc)`, the state reached by applying an action (only if inside the grid and not `#`) |
| **Goal test** | `if s == goal:` right after `heappop` |
| **g(n)** | `g_s` (stored in each frontier entry) and `best_g[s]`; computed by `g2 = g_s + step_cost` |
| **h(n)** | The function argument `h` (default `manhattan`), called as `h(s2, goal)` when a state is pushed |
| **f(n)** | `g2 + h(s2, goal)`, the first element of the tuple pushed on the heap; the heap orders by it |
| **Frontier** | `frontier`, a binary min-heap managed with `heapq.heappush` / `heappop` |
| **Visited states** | `best_g` (dictionary of the cheapest known cost per state) together with the stale-entry check `if g_s > best_g[s]: continue` |
| **Path reconstruction** | `parent` dictionary and `reconstruct(parent, state)`, which follows parents from the goal back to `None` and reverses the list |

**(a) Data structure for the frontier?** A priority queue implemented as a binary heap (a Python list managed by `heapq`). Entries are tuples `(f, tie_breaker, g, state)`.

**(b) How is the next state selected?** `heapq.heappop` always returns the tuple with the smallest `f = g + h`; ties are broken by the smaller `tie_breaker`, i.e. the entry that was inserted earlier.

**(c) Where is the heuristic calculated?** In `manhattan(s, goal)` (passed as `h`). It is evaluated once per generated state inside the successor loop (`h(s2, goal)`), and once for the start state.

**(d) Is $f(n)=g(n)+h(n)$ calculated explicitly?** Yes, in `g2 + h(s2, goal)` when a state is pushed. It is not stored separately per node; it is stored only as the heap priority (the first tuple element).

**(e) How is unnecessary repeated exploration prevented?** Through `best_g`: a successor is pushed only if it has not been reached before or it is reached with a *strictly smaller* `g`. Outdated duplicates that remain in the heap are discarded on removal with `if g_s > best_g[s]: continue`. With a consistent heuristic (such as Manhattan) each state is therefore expanded at most once; with an inconsistent one a state may be legitimately re-opened (see Task 6).

## Task 5: Compare A* with blind search (BFS)

The `bfs` function from Task 3 uses the identical `successors`, parent table and goal-test convention, so the only difference is the order of expansion: BFS uses a FIFO queue (shallowest first, ignoring the goal's location), A* uses a priority queue ordered by $g+h$. The warehouse is **not** changed.

In [6]:
grid, start, goal = parse_map(WAREHOUSE)
rb, ra = bfs(grid, start, goal), astar(grid, start, goal)

free_cells = sum(ch != "#" for row in grid for ch in row)
print(f"free cells (incl. S and G): {free_cells}\n")
print(f"{'Measure':18s}{'BFS':>8s}{'A*':>8s}")
print(f"{'Solution found':18s}{str(rb['found']):>8s}{str(ra['found']):>8s}")
print(f"{'Path length':18s}{rb['length']:>8d}{ra['length']:>8d}")
print(f"{'States expanded':18s}{rb['expanded']:>8d}{ra['expanded']:>8d}")
print("\nSame path?", rb["path"] == ra["path"])
assert rb["expanded"] == free_cells - 1 == ra["expanded"]      # both visit every free cell except the goal

bfs_exp_w, astar_exp_w, bfs_len_w, astar_len_w = rb["expanded"], ra["expanded"], rb["length"], ra["length"]

free cells (incl. S and G): 64

Measure                BFS      A*
Solution found        True    True
Path length             40      40
States expanded         63      63

Same path? True


On the original warehouse the two algorithms do exactly the same work: both find a path of the same length, and both expand 63 states, i.e. every free cell except the goal (64 free cells in total). The warehouse is a long winding corridor with a few dead ends, so there is nothing for a heuristic to prune: any algorithm has to walk the corridor. In addition, Manhattan distance is a poor estimate here (from `S` it says 20 moves; the true distance is 40), because walls force the robot to move *away* from the goal for long stretches.

To answer "why might A* expand fewer states?" properly we need a map where the goal's direction is informative, so we add an **open room** (a wall block between `S` and `G`) and also average over many random maps (both extra experiments leave the warehouse itself unchanged).

In [7]:
OPEN_ROOM = """
###############
#.............#
#.............#
#.......#.....#
#S......#....G#
#.......#.....#
#.............#
###############
"""
g, s, t = parse_map(OPEN_ROOM)
ob, oa = bfs(g, s, t), astar(g, s, t)
print(f"Open room  ->  BFS: length {ob['length']}, expanded {ob['expanded']}   |   A*: length {oa['length']}, expanded {oa['expanded']}")
print(render(g, oa["path"]))

bfs_exp_o, astar_exp_o = ob["expanded"], oa["expanded"]

# Average over the 300 random maps (solvable ones only)
tot_b = tot_a = n = 0
for g, s, t in RANDOM_MAPS:
    b, a = bfs(g, s, t), astar(g, s, t)
    if b["found"]:
        n += 1; tot_b += b["expanded"]; tot_a += a["expanded"]
mean_b, mean_a = round(tot_b / n, 1), round(tot_a / n, 1)
print(f"\nRandom 8x8 maps ({n} solvable): mean expanded BFS = {mean_b}, A* = {mean_a}")
assert bfs_exp_o > astar_exp_o and mean_b > mean_a

Open room  ->  BFS: length 16, expanded 74   |   A*: length 16, expanded 61
###############
#.............#
#......***....#
#......*#*....#
#S******#****G#
#.......#.....#
#.............#
###############

Random 8x8 maps (261 solvable): mean expanded BFS = 13.1, A* = 6.7


**(a) Did both algorithms find a solution?** Yes, on every solvable map.

**(b) Same path length?** Yes: 40 moves for BFS and 40 for A* on the warehouse. BFS is optimal for unit costs, and A* with an admissible heuristic is optimal too, so equal lengths are expected (the specific paths may differ when several shortest paths exist).

**(c) Which expanded fewer states?** On the warehouse: neither (63 vs. 63). In the open room A* expanded 61 states versus 74 for BFS, and on random maps A* averaged 6.7 against 13.1 for BFS.

**(d) Why might A\* expand fewer states?** BFS expands states in order of distance from the start, in all directions, without using any knowledge about where the goal is. A* adds the estimate $h(n)$ of the remaining cost, so states that are *near the start but pointing away from the goal* get a large $f=g+h$ and are postponed or never expanded. The saving exists only if $h$ discriminates between good and bad directions; in the corridor it does not.

**Think about it: what information does the algorithm use to decide where to search next?** BFS uses only the depth (cost so far). A* uses cost so far plus a domain-specific estimate of the cost to go. The benefit is entirely determined by the quality of that estimate, so A* is not better simply for being "more intelligent".

## Task 6: Investigate the heuristic

**Why Manhattan distance suits this warehouse.** With four moves of cost 1, any path from $(x,y)$ to $(x_G,y_G)$ needs at least $|x-x_G|$ vertical and $|y-y_G|$ horizontal unit steps, even with no obstacles. Obstacles can only make the true cost $h^*(n)$ larger, never smaller, so $h(n)=|x-x_G|+|y-y_G|\le h^*(n)$ for every cell: Manhattan distance is **admissible** (and, since one move changes it by at most 1, also **consistent**).

We now test four variants on the same warehouse, plus the exact cost-to-go $h^*$ (computed by BFS from the goal) as a "perfect" reference. For each we also count how many cells violate admissibility ($h>h^*$) and how many states were expanded more than once.

In [8]:
def true_dist(grid, goal):
    """h*(n): exact number of moves from every reachable cell to the goal (BFS from the goal)."""
    dist, q = {goal: 0}, deque([goal])
    while q:
        s = q.popleft()
        for _, s2, _ in successors(grid, s):
            if s2 not in dist:
                dist[s2] = dist[s] + 1; q.append(s2)
    return dist

grid, start, goal = parse_map(WAREHOUSE)
hstar = true_dist(grid, goal)

HEURISTICS = {
    "h = 0":            zero,
    "Manhattan":        manhattan,
    "Euclidean":        euclidean,
    "2 x Manhattan":    lambda s, t: 2 * manhattan(s, t),
    "exact h* (oracle)": lambda s, t, d=hstar: d.get(s, 0),
}

print(f"{'heuristic':20s}{'found':>7s}{'length':>8s}{'expanded':>10s}{'re-expanded':>13s}{'h>h* cells':>12s}")
warehouse_rows = {}
for name, h in HEURISTICS.items():
    r = astar(grid, start, goal, h)
    viol = sum(h(s, goal) > d + 1e-9 for s, d in hstar.items())
    warehouse_rows[name] = (r["found"], r["length"], r["expanded"], r["expanded"] - r["distinct"], viol)
    print(f"{name:20s}{str(r['found']):>7s}{r['length']:>8d}{r['expanded']:>10d}{r['expanded'] - r['distinct']:>13d}{viol:>12d}")

z_exp, m_exp, e_exp, w2_exp, o_exp = (warehouse_rows[k][2] for k in HEURISTICS)
w2_reexp, w2_viol = warehouse_rows["2 x Manhattan"][3], warehouse_rows["2 x Manhattan"][4]
w2_len = warehouse_rows["2 x Manhattan"][1]
print("\nManhattan estimate at S:", manhattan(start, goal), "  true cost h*(S):", hstar[start])

heuristic             found  length  expanded  re-expanded  h>h* cells
h = 0                  True      40        63            0           0
Manhattan              True      40        63            0           0
Euclidean              True      40        63            0           0
2 x Manhattan          True      40        66            3          18
exact h* (oracle)      True      40        40            0           0

Manhattan estimate at S: 20   true cost h*(S): 40


**Reading the warehouse table.** On this single map, h = 0, Manhattan and Euclidean all expand the same 63 states, because (as in Task 5) the corridor leaves nothing to prune. Multiplying Manhattan by 2 made the heuristic inadmissible on 18 cells and led to 66 expansions, i.e. 3 re-expansions of states that had been expanded before. The path is still optimal here (40), but only by luck of the map: there is essentially only one route. Even the exact oracle $h^*$ expands 40 states, far fewer, which shows how much a *good* heuristic could save, and that Manhattan is far from $h^*$ in a maze.

One map cannot reveal what an over-optimistic heuristic does to optimality, so we test it on a small map with a detour, and then on many random open maps.

In [9]:
OVERSHOOT = """
#######
#.....#
##.#S.#
#...#.#
###...#
#G....#
#######
"""
g, s, t = parse_map(OVERSHOOT)
opt = bfs(g, s, t)
w2 = astar(g, s, t, HEURISTICS["2 x Manhattan"])
print("Optimal (BFS)       length:", opt["length"])
print("A* with Manhattan   length:", astar(g, s, t)["length"])
print("A* with 2 x Manhattan length:", w2["length"], "<- longer than optimal!")
print("\nmap with the 2 x Manhattan path (*):")
print(render(g, w2["path"]))
overshoot_opt, overshoot_w2 = opt["length"], w2["length"]

# Many random open maps: how often is each heuristic suboptimal and how much work does it take?
OPEN_MAPS = make_maps(seed=3, n=500, rows=10, cols=10, density=0.15)
variants = {"h = 0": zero, "Manhattan": manhattan, "Euclidean": euclidean,
            "1.5 x Manhattan": lambda s, t: 1.5 * manhattan(s, t),
            "2 x Manhattan": lambda s, t: 2 * manhattan(s, t),
            "5 x Manhattan": lambda s, t: 5 * manhattan(s, t)}
print(f"\n500 random 10x10 maps: suboptimal paths and mean states expanded")
print(f"{'heuristic':18s}{'suboptimal':>12s}{'mean expanded':>16s}")
stats = {}
for name, h in variants.items():
    sub = tot = n = 0
    for g, s, t in OPEN_MAPS:
        ref = bfs(g, s, t)
        if not ref["found"]: continue
        r = astar(g, s, t, h)
        n += 1; tot += r["expanded"]; sub += (r["length"] > ref["length"])
        assert r["found"]
    stats[name] = (sub, round(tot / n, 1), n)
    print(f"{name:18s}{sub:>8d}/{n:<3d}{stats[name][1]:>16}")

sub5, sub2 = stats["5 x Manhattan"][0], stats["2 x Manhattan"][0]
n_open = stats["Manhattan"][2]
exp0, expM, expE, exp5 = stats["h = 0"][1], stats["Manhattan"][1], stats["Euclidean"][1], stats["5 x Manhattan"][1]
assert stats["Manhattan"][0] == 0 and stats["Euclidean"][0] == 0 and stats["h = 0"][0] == 0
assert sub5 > 0 and exp0 > expM > exp5

Optimal (BFS)       length: 8
A* with Manhattan   length: 8
A* with 2 x Manhattan length: 10 <- longer than optimal!

map with the 2 x Manhattan path (*):
#######
#.***.#
##*#S.#
#.**#.#
###*..#
#G**..#
#######

500 random 10x10 maps: suboptimal paths and mean states expanded
heuristic           suboptimal   mean expanded
h = 0                    0/489            28.3
Manhattan                0/489            11.4
Euclidean                0/489            12.4
1.5 x Manhattan          1/489             6.7
2 x Manhattan            7/489             6.7
5 x Manhattan           29/489             6.6


### Results of the heuristic investigation

| Heuristic | Solution found? | Path length (warehouse) | States expanded (warehouse) | Optimal? |
|---|---|---|---|---|
| h = 0 | yes | 40 | 63 | always (admissible) |
| Manhattan | yes | 40 | 63 | always (admissible, consistent) |
| Euclidean | yes | 40 | 63 | always (admissible) |
| 2 × Manhattan | yes | 40 | 66 | **not guaranteed** (the trap map: 10 instead of 8) |

**1. h = 0.** The search is still correct and always optimal, since $0\le h^*$ is admissible, but with no directional information A* becomes uniform-cost search (Dijkstra), which for unit costs expands states in the same breadth-first manner as BFS. On 10×10 random maps it averaged 28.3 expansions against 11.4 for Manhattan.

**2. Euclidean.** It is admissible (the straight-line distance never exceeds the grid distance) and also finds optimal paths, but it is *less informed* than Manhattan on a 4-connected grid, because $\sqrt{\Delta x^2+\Delta y^2}\le\Delta x+\Delta y$. A smaller admissible heuristic means more states are expanded (random maps: 12.4 versus 11.4 for Manhattan).

**3. Multiplied by 2.** It guides the search more aggressively, which is why it expanded fewer states on open maps (6.6 on average for 5× versus 11.4 for Manhattan), but it **overestimates** $h^*$ in places, so A* can commit to a route that looks promising and return it before a cheaper route is examined. The trap map returned a 10-move path where 8 is possible, and on the 489 solvable random maps ×2 was suboptimal 7 times and ×5 29 times (Manhattan and Euclidean never). The price for speed is a loss of the optimality guarantee; in the warehouse, an inconsistent heuristic also caused re-expansions.

**Think about it: too optimistic vs. too aggressive.** An *optimistic* (admissible) heuristic never overestimates, so A* can trust the first time it reaches the goal: the cheaper-looking alternatives have all been examined, because their $f$ was ≤ the goal's cost. The smaller the heuristic, the more such alternatives must be examined (h = 0 is the extreme: a lot of work, optimal answer). An *aggressive* heuristic (overestimating) makes A* greedy: fast, but optimality is lost. The experiments show this tradeoff directly: work decreases as $h$ grows, and suboptimal answers appear once $h$ exceeds $h^*$.

## Task 7: Evaluate the LLM-generated agent

*The answers below are a draft based on the development of this notebook. Because this question is about your own session with the LLM, edit them so that they describe what actually happened when you ran it (your prompts, the code you received, and the changes you made).*

**Who did what:**

| | Content |
|---|---|
| **I designed** | The problem formulation (Task 0), the agent design (Task 1: tuple states, `successors`, heap of `(f, tie, g, state)`, `parent` table, goal test on removal, the required outputs), and the test plan with expected results written before running |
| **The LLM suggested** | The code skeleton and explanations: the heap with a tie-breaker counter, `reconstruct`, the `parse_map` function, and the BFS variant |
| **I accepted** | The general structure of `astar`, the `successors` generator, the heuristic functions and BFS |
| **I changed / added** | `best_g` with the stale-entry check (so a cheaper path can re-open a state and inconsistent heuristics are handled correctly), storing `g` inside the heap entry, the distinct/total expansion counts, the independent `check_path` validator, and the heuristic as a parameter |
| **I tested** | Tests 1-4, random cross-validation against BFS, the three seeded-bug versions, and the heuristic experiments |

**1. What was correct immediately?** The overall A* loop, the use of `heapq`, the four-direction transition model, the goal test and the path reconstruction were right on the warehouse and on the small hand-made maps.

**2. Bugs or design problems?** The main design issue was the duplicate handling. The simplest "closed set on push" or "never re-open a state" versions look fine on the warehouse but are wrong in general (the closed-on-push bug in Task 3 returned a 10-move path where 8 is optimal), and versions without any visited set never terminate on unsolvable maps. Also, "states expanded" is ambiguous (goal included or not; goal test on push or on pop), which has to be fixed before A* and BFS can be compared fairly.

**3. How were problems discovered?** Not from "a path was found" on the warehouse: the closed-on-push version passed it, and the greedy version also returned a plausible-looking path (48 moves) that is only wrong when compared with the known optimum of 40. The faults showed up through the unsolvable map (non-termination), the trap map, and the random cross-validation against BFS.

**4. Terminology or data structures I did not understand?** The heap-entry tuple `(f, tie, g, state)` needs explanation: the counter stops Python from comparing states when `f` ties and makes ties first-in-first-out, and `heapq` has no decrease-key operation, which is why stale duplicates are skipped lazily instead of being updated in place.

**5. Did I modify the generated code?** Yes (see the table): tracking of the best `g`, the stale-entry check, the extra counters and the validator.

**6. Which tests were most useful?** The random cross-validation against an independent, simpler algorithm (BFS), which caught all three faulty versions, plus the trap map and the no-solution map. The hand-made tiny maps were the least discriminating, and the warehouse is only useful because we compare against the *known optimal length* (40), not just against "a path was found".

**7. Could I have trusted the program without testing it?** No. The faulty variants in Task 3 produce legitimate-looking paths, and the closed-on-push version passes the warehouse and all small hand-made tests; only targeted tests with known expected behaviour (trap map, unsolvable map, random cross-validation) reveal the faults.

**8. What did I understand about A\* that I did not understand before implementing it?** That A* is a *family* determined by the evaluation function: $h=0$ gives uniform-cost search, $f=h$ gives greedy search, and a heuristic that is too large gives fast but possibly suboptimal answers. Also that optimality depends on details such as testing the goal on removal and on handling of duplicate states, and that a heuristic helps only where it is informative (the corridor shows it cannot always help).

## Final reflection

**1. Why is it important to formulate the search problem before writing the search algorithm?**
The formulation $(S,A,T,s_0,G,c)$ is the specification that the program is later checked against. It settles what a state contains, which moves are legal, what the goal test and the costs are, and what counts as a solution. Without it there is nothing to decide whether a generated program is right, and the LLM's choices (how to represent the state, when to test for the goal, how to count "expanded") would silently become the definition of the problem. The generic algorithm (search) is reusable only because the problem-specific parts are isolated in `successors`, the goal test and the heuristic.

**2. In what sense is A\* an "informed" search algorithm?**
BFS and DFS use only the structure of the graph: they know nothing about how close a state is to the goal. A* additionally uses problem-specific knowledge in the form of a heuristic $h(n)$ that estimates the remaining cost, and orders the frontier by $f(n)=g(n)+h(n)$. The algorithm is "informed" because this extra information guides which state is expanded next. How much it helps depends on the quality of the information, as the open room (clear savings) and the warehouse corridor (none) show.

**3. Why does the choice of heuristic matter?**
It affects both correctness and efficiency. An admissible heuristic ($h\le h^*$) keeps A* optimal, and among admissible heuristics a larger one (closer to $h^*$) expands fewer states: h = 0 < Euclidean < Manhattan < exact $h^*$ in informativeness. A heuristic that overestimates may make the search faster but can return suboptimal paths (the trap map), and an inconsistent one can cause states to be re-expanded. A poorly chosen heuristic can also make A* no better than blind search, as on the warehouse.

**4. What did the LLM contribute to the engineering process?**
It translated a clear design into a working first version quickly: the boilerplate (map parsing, heap handling, path reconstruction), the BFS comparison variant, and explanations of unfamiliar constructs such as the heap tie-breaker. That saved time that could be spent on testing and interpretation. It did not supply the problem formulation, the expected results for the tests, or the judgement about what counts as evidence; those are the engineer's contribution.

**5. What could go wrong if an engineer simply accepted LLM-generated code without testing it?**
The code might look right and pass the obvious example while being wrong in general. In this notebook, three plausible faulty versions returned suboptimal paths (greedy priority, closing states too early) or never terminated (no visited set); the closed-on-push version even passed the original warehouse and all the small hand-made tests, and was exposed only by the trap map and the random cross-validation. In a real system such a fault would surface as a wasteful route, a hung program, or an unsafe plan, far from where it was introduced. The lesson of the lab is the same: working output ≠ validated algorithm; the engineer stays responsible for understanding, testing and validating the system.

## Appendix: prompts used with the LLM

**Prompt 1 (A\* implementation):** as quoted in Task 2.

**Prompt 2 (BFS for the comparison, Task 5):**
> Using the same `parse_map`, `successors` and `reconstruct` functions and the same result dictionary, write a BFS version of the agent with a `collections.deque` frontier and a `parent` dictionary as the reached table. Test for the goal when a state is removed from the queue, exactly as in the A* version, and count expanded states the same way.

**Prompt 3 (explanation for Task 6):**
> Explain why Manhattan distance is an admissible and consistent heuristic for a 4-connected grid with unit move costs.

*(Replace these with your actual prompts if they differ; the handout asks for the important prompts to be included.)*

**Reproducibility:** Python 3, standard library only; random maps use `random.Random(seed)` with seeds 1 (8×8, density 0.25, 300 maps) and 3 (10×10, density 0.15, 500 maps). Run all cells in order.